# 02 – Merge, Group Split (by user), Causal Crop & Scaling


This notebook takes the chunks generated in **01_build_sequences**, executes:
1) **Merge** the chunks into `X_final.npy`/ `y_final.npy` (if not already present)
2) **Group split per utente** (niente leakage)
3) **Causal crop**: only keeps time steps up to the **event slot** (inclusive)
4) **Scaling 3D** fit **solo su train** (nan‑safe, imputation a mean train)
5) Saving sets **train/val/test** and scaler → ready for LSTM




In [2]:
# %% 0) Setup paths
import os, glob, numpy as np, pandas as pd

# folder that contains X_chunk_*.npy / y_chunk_*.npy / meta_chunk_*.npz
SAVE_DIR = "process_data/event_sequences_chunks"  # ← adatta se necessario (es. run_YYYYMMDD_HHMM)

# folder where to save the sets ready for training
OUT_DIR = "process_data/processed_lstm"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 1) Utility: loaad chunk + merge + meta
 Load `X`, `y`, `user_ids` and read the list `features` from the meta.

In [3]:
# loader with extra meta consistent with 01

def load_with_meta(save_dir):
    """
    It reads X/Y chunks and, if present, meta .npcs produced by 01.
    Returns: x, y, user_ids, features, META (dict with keys present in meta
    """
    import glob, os
    xps = sorted(glob.glob(os.path.join(save_dir, "X_chunk_*.npy")))
    yps = sorted(glob.glob(os.path.join(save_dir, "y_chunk_*.npy")))
    mps = [xp.replace("X_chunk_", "meta_chunk_").replace(".npy", ".npz") for xp in xps]
    assert len(xps) == len(yps) == len(mps) and xps, "Chunk mancanti"

    Xs, Ys, U = [], [], []
    META = {}         
    features = None

    for xp, yp, mp in zip(xps, yps, mps):
        Xs.append(np.load(xp).astype(np.float32))
        Ys.append(np.load(yp).astype(np.int8).reshape(-1))

        z = np.load(mp, allow_pickle=True)

        # feature list (already WITHOUT label, because 01 saves X without 'y_override')
        if features is None and "features" in z.files:
            features = z["features"].tolist()

        # user_ids (se presente)
        if "user_ids" in z.files:
            U.append(np.array(z["user_ids"]))

        # gather any extra columns if present
        for k in ("event_times", "actual_event", "reward", "appliance", "city"):
            if k in z.files:
                META.setdefault(k, []).append(np.array(z[k]))

    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(Ys, axis=0)
    user_ids = np.concatenate(U, axis=0) if U else None
    META = {k: np.concatenate(v, axis=0) for k, v in META.items()}
    return X, y, user_ids, features, META


def load_Xy_final_or_chunks(save_dir):
    """
    if X_final/y_final exist, use them,
    otherwise reads from chunks (with meta).
    """
    x_final = os.path.join(save_dir, "X_final.npy")
    y_final = os.path.join(save_dir, "y_final.npy")
    if os.path.exists(x_final) and os.path.exists(y_final):
        X = np.load(x_final).astype(np.float32)
        y = np.load(y_final).astype(np.int8).reshape(-1)
        _, _, user_ids, feats, META = load_with_meta(save_dir)  # meta dai chunk
        return X, y, user_ids, feats, META
    else:
        return load_with_meta(save_dir)

# CALL:
X, y, user_ids, features, META = load_Xy_final_or_chunks(SAVE_DIR)
print(f"Caricato: X{X.shape}, y{y.shape}, user_ids={None if user_ids is None else len(user_ids)}")
print("features:", features)



Caricato: X(255893, 48, 12), y(255893,), user_ids=255893
features: ['energy_Wh', 'reward_rate', 'notice_time', 'week_in_trial', 'hour_sin', 'hour_cos', 'temperature', 'wind_u', 'wind_v', 'precip_mm_log', 'ssrd_kwh', 'snr_kwh']


## 2) No `override` inside input
`override` was only used to create `y` at the event slot. It must not enter the model's input channels.

In [4]:
# --- robust label + input features list ---
meta0 = np.load(sorted(glob.glob(os.path.join(SAVE_DIR, "meta_chunk_*.npz")))[0], allow_pickle=True)
label_col = str(meta0.get("label_col", "y_override"))

if label_col in features:
    idx_over = features.index(label_col)
    features_input = [f for i, f in enumerate(features) if i != idx_over]
    X = X[..., [i for i in range(X.shape[-1]) if i != idx_over]]  # togli il canale label
else:
    # The label has already been removed upstream, I use features as they are
    features_input = features

print("Label column:", label_col)
print("Input features:", features_input)


Label column: y_override
Input features: ['energy_Wh', 'reward_rate', 'notice_time', 'week_in_trial', 'hour_sin', 'hour_cos', 'temperature', 'wind_u', 'wind_v', 'precip_mm_log', 'ssrd_kwh', 'snr_kwh']


## 3) Group split by user (train/val/test)


In [5]:
# %% 3) Split by user
from sklearn.model_selection import GroupShuffleSplit

def group_split(X, y, groups, train_size=0.7, val_size=0.15, seed=42):
    assert groups is not None, "Mancano user_ids per il group split"
    gss1 = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    tr, tmp = next(gss1.split(X, y, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, train_size=val_size/(1-train_size), random_state=seed)
    v, te = next(gss2.split(X[tmp], y[tmp], groups=groups[tmp]))
    v, te = tmp[v], tmp[te]
    return tr, v, te

if user_ids is None:
    print(" user_ids assenti: eseguo uno split casuale (NON per risultati finali).")
    n = len(X); idx = np.random.permutation(n)
    ntr = int(0.7*n); nv = int(0.15*n)
    idx_tr, idx_v, idx_te = idx[:ntr], idx[ntr:ntr+nv], idx[ntr+nv:]
else:
    user_ids = np.asarray(user_ids)
    idx_tr, idx_v, idx_te = group_split(X, y, user_ids, seed=SEED)

len_tr, len_v, len_te = len(idx_tr), len(idx_v), len(idx_te)
print(f"Split → train={len_tr}, val={len_v}, test={len_te}")

Split → train=185156, val=35248, test=35489


## 4) Causal crop (no futuro) e Scaling 3D (fit solo su train)
I use a `StandardScaler3D` that is **nan‑safe**: it calculates the mean/std on the train set ignoring the NaNs and then imputes the NaNs with the mean of the train set.

In [6]:
# %% 4) Crop causale + scaler nan-safe
center = X.shape[1] // 2
def causal_crop(A):
    return A[:, :center+1, :].copy()

X_tr_raw, y_tr = causal_crop(X[idx_tr]), y[idx_tr]
X_v_raw,  y_v  = causal_crop(X[idx_v]),  y[idx_v]
X_te_raw, y_te = causal_crop(X[idx_te]), y[idx_te]

class StandardScaler3D:
    def fit(self, X):
        X2 = X.reshape(-1, X.shape[-1])
        self.mean_ = np.nanmean(X2, axis=0)
        self.std_  = np.nanstd(X2, axis=0) + 1e-8
        return self
    def transform(self, X):
        Z = (X - self.mean_) / self.std_
        # imputa eventuali NaN con 0 (cioè mean dopo standardizzazione)
        nanmask = np.isnan(Z)
        if nanmask.any():
            Z[nanmask] = 0.0
        return Z
    def fit_transform(self, X):
        return self.fit(X).transform(X)

scaler = StandardScaler3D()
X_tr = scaler.fit_transform(X_tr_raw)
X_v  = scaler.transform(X_v_raw)
X_te = scaler.transform(X_te_raw)

def summarize_split(name, Xs, ys):
    n = len(ys); pos = int(ys.sum()); rate = pos/max(n,1)
    print(f"{name:<5} → N={n:<6} Pos={pos:<6} ({rate:>6.2%}) | X{Xs.shape}")

summarize_split("train", X_tr, y_tr)
summarize_split("val",   X_v,  y_v)
summarize_split("test",  X_te, y_te)

# pos_weight per la loss
pos = max(int(y_tr.sum()), 1)
neg = int((y_tr==0).sum())
pos_weight = neg / pos
print(f"pos_weight (train) ≈ {pos_weight:.3f}")

train → N=185156 Pos=10759  ( 5.81%) | X(185156, 25, 12)
val   → N=35248  Pos=2305   ( 6.54%) | X(35248, 25, 12)
test  → N=35489  Pos=1693   ( 4.77%) | X(35489, 25, 12)
pos_weight (train) ≈ 16.209


In [8]:
# split QC
uids = np.asarray(user_ids)
tr_u = set(uids[idx_tr]); v_u = set(uids[idx_v]); te_u = set(uids[idx_te])
print("overlap train-val:", len(tr_u & v_u),
      "| train-test:", len(tr_u & te_u),
      "| val-test:", len(v_u & te_u))

def summarize(y, name):
    pos = int(y.sum()); n = len(y)
    print(f"{name}: N={n} | Pos={pos} ({pos/n:0.2%})")
summarize(y_tr, "train"); summarize(y_v, "val"); summarize(y_te, "test")

print("NaN in X_train:", np.isnan(X_tr).any(),
      "| X_val:", np.isnan(X_v).any(),
      "| X_test:", np.isnan(X_te).any())


overlap train-val: 0 | train-test: 0 | val-test: 0
train: N=185156 | Pos=10759 (5.81%)
val: N=35248 | Pos=2305 (6.54%)
test: N=35489 | Pos=1693 (4.77%)
NaN in X_train: False | X_val: False | X_test: False


In [9]:
for name,y in [("train",y_tr),("val",y_v),("test",y_te)]:
    print(name, "pos_rate:", float(y.mean()))


train pos_rate: 0.05810775778262654
val pos_rate: 0.06539378120744439
test pos_rate: 0.04770492265208938


## 5) Backups (set + scaler + config)
Save the 3 sets, the scaler (mean/std) and a `config.json` with the basic information.

In [11]:
# %% 5) Save all
ts = pd.Timestamp.now().strftime("run_%Y%m%d_%H%M")
OUT_RUN = os.path.join(OUT_DIR, ts)
os.makedirs(OUT_RUN, exist_ok=True)

np.save(os.path.join(OUT_RUN, "X_train.npy"), X_tr.astype(np.float32))
np.save(os.path.join(OUT_RUN, "y_train.npy"), y_tr.astype(np.int8))
np.save(os.path.join(OUT_RUN, "X_val.npy"),   X_v.astype(np.float32))
np.save(os.path.join(OUT_RUN, "y_val.npy"),   y_v.astype(np.int8))
np.save(os.path.join(OUT_RUN, "X_test.npy"),  X_te.astype(np.float32))
np.save(os.path.join(OUT_RUN, "y_test.npy"),  y_te.astype(np.int8))

np.savez(os.path.join(OUT_RUN, "scaler.npz"), mean=scaler.mean_, std=scaler.std_)

# --- SAFETY: rebuild features_input if it does not exist ---
try:
    features_input  # defined in cell "drop label"
except NameError:
    # label_col should have been read by meta; fallback to 'y_override'
    label_col = locals().get("label_col", "y_override")
    # 'features' è la lista originale (con la label); togli la label se presente
    features_input = [f for f in features if f != label_col]


cfg = {
    "seed": SEED,
    "save_dir_chunks": SAVE_DIR,
    "out_dir": OUT_RUN,
    "input_features": features_input,
    "sequence_length_total": int(X.shape[1]),
    "sequence_length_used": int(X_tr.shape[1]),
    "center_index": int(X.shape[1]//2),
    "pos_weight_train": float(pos_weight),
    "sizes": {
        "train": int(len(y_tr)),
        "val": int(len(y_v)),
        "test": int(len(y_te))
    }
}


cfg.update({
    "label_col": label_col,                       # es. "y_override"
    "input_features": features_input,             # lista senza la label
    "all_features_in_chunk": features,            # lista originaria (con label)
    "meta_keys": sorted(list(META.keys() or [])), # es. user_ids/event_times/...
    "center_idx": int(X.shape[1]//2),             # t=0 della finestra
    "run_info": {
        "created_at": pd.Timestamp.now().isoformat(),
        "save_dir_chunks": SAVE_DIR
    }
})

# (opzionale utile) salva anche gli indici dello split per tracciabilità
np.save(os.path.join(OUT_RUN, "idx_train.npy"), idx_tr.astype(np.int64))
np.save(os.path.join(OUT_RUN, "idx_val.npy"),   idx_v.astype(np.int64))
np.save(os.path.join(OUT_RUN, "idx_test.npy"),  idx_te.astype(np.int64))



with open(os.path.join(OUT_RUN, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("\nSalvato tutto in:", OUT_RUN)


Salvato tutto in: process_data/processed_lstm/run_20250826_1904


se ci sono problemi

In [ ]:
cfg = {
    "seed": SEED,
    "save_dir_chunks": SAVE_DIR,
    "out_dir": OUT_RUN,

    # elenco input al modello (senza label)
    "input_features": features_input,

    # per riproducibilità: elenco originale dal chunk (con label)
    "all_features_in_chunk": features,

    "sequence_length_total": int(X.shape[1]),
    "sequence_length_used":  int(X_tr.shape[1]),
    "center_index":          int(X.shape[1] // 2),

    "pos_weight_train": float(pos_weight),

    "sizes": {
        "train": int(len(y_tr)),
        "val":   int(len(y_v)),
        "test":  int(len(y_te)),
    },

    # quali meta abbiamo (chiavi)
    "meta_keys": sorted(list(META.keys())) if isinstance(META, dict) else [],

    "run_info": {
        "created_at": pd.Timestamp.now().isoformat(),
        "save_dir_chunks": SAVE_DIR
    },
}


### Next step
Open **03_lstm.ipynb** and use the files saved in `data/processed_lstm/<run>/`.